# Unit 7 Lecture 1: Numerical ODE Integration (Student)

## Objectives
- Understand numerical integration methods for ODEs
- Implement Euler's method and Runge-Kutta methods
- Compare accuracy and computational cost of different methods
- Use scipy's solve_ivp for production simulations
- Analyze step size effects and stability

## Why This Matters
Most dynamic systems cannot be solved analytically. Numerical integration is essential for:
- Nonlinear systems (pendulum, robot dynamics)
- Multi-body systems (cars, aircraft, mechanisms)
- Systems with complex forcing functions
- Real-time simulation and control

## Overview
**Part A**: Euler's Method - simplest integration scheme  
**Part B**: Runge-Kutta Methods - higher accuracy  
**Part C**: scipy solve_ivp - production-ready solvers  
**Duration**: ~90 minutes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import time

# Configure plotting
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

## Part A: Euler's Method

### The ODE Problem
Given a first-order ODE system:
$$\frac{d\mathbf{y}}{dt} = \mathbf{f}(t, \mathbf{y})$$

with initial condition $\mathbf{y}(t_0) = \mathbf{y}_0$, we want to find $\mathbf{y}(t)$ for $t > t_0$.

### Forward Euler Method
The simplest numerical integration scheme:
$$\mathbf{y}_{n+1} = \mathbf{y}_n + h \cdot \mathbf{f}(t_n, \mathbf{y}_n)$$

where $h$ is the step size (timestep).

**Properties**:
- **Order**: First-order accurate, $O(h)$
- **Stability**: Conditionally stable (small $h$ required)
- **Pros**: Simple to implement, fast per step
- **Cons**: Low accuracy, can be unstable

### Example System: Spring-Mass-Damper
$$m\ddot{x} + c\dot{x} + kx = 0$$

Convert to first-order system:
$$\begin{bmatrix} \dot{x} \\ \dot{v} \end{bmatrix} = \begin{bmatrix} v \\ -\frac{c}{m}v - \frac{k}{m}x \end{bmatrix}$$

Let's implement and compare with analytical solution.

In [ ]:
# Example 1: Spring-Mass-Damper with Euler's Method
print("="*70)
print("Example 1: Euler's Method for Spring-Mass-Damper")
print("="*70)

# System parameters
m = 1.0      # kg
k = 100.0    # N/m
c = 2.0      # N·s/m (underdamped)
omega_n = np.sqrt(k/m)
zeta = c / (2*np.sqrt(m*k))

print(f"\nSystem parameters:")
print(f"Natural frequency: ω_n = {omega_n:.3f} rad/s")
print(f"Damping ratio: ζ = {zeta:.4f} (underdamped)")

# Define ODE function
def spring_mass_damper(t, y):
    """Spring-mass-damper ODE: dy/dt = f(t,y)"""
    x, v = y
    dxdt = v
    dvdt = -(c/m)*v - (k/m)*x
    return np.array([dxdt, dvdt])

# Initial conditions
x0 = 1.0    # m
v0 = 0.0    # m/s
y0 = np.array([x0, v0])

# Time span
t_end = 2.0
h_values = [0.1, 0.05, 0.01]  # Different step sizes

# Analytical solution for comparison
omega_d = omega_n * np.sqrt(1 - zeta**2)
t_exact = np.linspace(0, t_end, 1000)
x_exact = x0 * np.exp(-zeta*omega_n*t_exact) * np.cos(omega_d*t_exact)

# Euler's method implementation
def euler_method(f, y0, t_span, h):
    """Forward Euler integration"""
    t0, tf = t_span
    t = np.arange(t0, tf + h/2, h)
    n_steps = len(t)
    
    y = np.zeros((n_steps, len(y0)))
    y[0] = y0
    
    for i in range(n_steps - 1):
        y[i+1] = y[i] + h * f(t[i], y[i])
    
    return t, y

# Compare different step sizes
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, h in enumerate(h_values):
    # Run Euler method
    t_euler, y_euler = euler_method(spring_mass_damper, y0, (0, t_end), h)
    x_euler = y_euler[:, 0]
    
    # Calculate error
    x_exact_interp = np.interp(t_euler, t_exact, x_exact)
    error = np.abs(x_euler - x_exact_interp)
    max_error = np.max(error)
    
    print(f"\nStep size h = {h:.3f}:")
    print(f"Number of steps: {len(t_euler)}")
    print(f"Max error: {max_error:.6f} m")
    
    # Plot position
    ax = axes[0, 0] if idx == 0 else axes[0, 1] if idx == 1 else axes[1, 0]
    ax.plot(t_exact, x_exact, 'k-', linewidth=2, label='Exact', alpha=0.7)
    ax.plot(t_euler, x_euler, 'ro-', markersize=4, label=f'Euler (h={h})')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Position (m)')
    ax.set_title(f'Euler Method: h = {h:.3f}\nMax Error = {max_error:.6f} m')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Error comparison
axes[1, 1].loglog(h_values, [0.1, 0.05, 0.01], 'bo-', markersize=8, linewidth=2, label='Error ∝ h')
axes[1, 1].set_xlabel('Step Size h (s)')
axes[1, 1].set_ylabel('Max Error (m)')
axes[1, 1].set_title('Error vs Step Size (log-log)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('notebooks/Teacher/figs/u7_l1_euler_method.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n{'='*70}")
print("Key Observations:")
print("• Euler method shows phase lag (delayed oscillation)")
print("• Error decreases with smaller step size")
print("• First-order method: error ∝ h")
print("• Requires small h for accuracy")
print("="*70)
# TODO: Use these parameters in your solutions below


## Part B: Runge-Kutta Methods

### Higher-Order Accuracy
Runge-Kutta methods achieve higher accuracy by evaluating the derivative at multiple points.

### RK2 (Midpoint Method)
$$\begin{align}
\mathbf{k}_1 &= h \cdot \mathbf{f}(t_n, \mathbf{y}_n) \\
\mathbf{k}_2 &= h \cdot \mathbf{f}\left(t_n + \frac{h}{2}, \mathbf{y}_n + \frac{\mathbf{k}_1}{2}\right) \\
\mathbf{y}_{n+1} &= \mathbf{y}_n + \mathbf{k}_2
\end{align}$$

**Order**: Second-order accurate, $O(h^2)$

### RK4 (Classical Fourth-Order)
$$\begin{align}
\mathbf{k}_1 &= h \cdot \mathbf{f}(t_n, \mathbf{y}_n) \\
\mathbf{k}_2 &= h \cdot \mathbf{f}\left(t_n + \frac{h}{2}, \mathbf{y}_n + \frac{\mathbf{k}_1}{2}\right) \\
\mathbf{k}_3 &= h \cdot \mathbf{f}\left(t_n + \frac{h}{2}, \mathbf{y}_n + \frac{\mathbf{k}_2}{2}\right) \\
\mathbf{k}_4 &= h \cdot \mathbf{f}(t_n + h, \mathbf{y}_n + \mathbf{k}_3) \\
\mathbf{y}_{n+1} &= \mathbf{y}_n + \frac{1}{6}(\mathbf{k}_1 + 2\mathbf{k}_2 + 2\mathbf{k}_3 + \mathbf{k}_4)
\end{align}$$

**Order**: Fourth-order accurate, $O(h^4)$

**Trade-offs**:
- RK4 requires 4 function evaluations per step
- Much more accurate than Euler for same step size
- Standard choice for many engineering applications

In [ ]:
# TODO: Create visualization# Hint: Use matplotlib to plot your results## Suggested structure:# 1. Create figure and axes# 2. Plot calculated results# 3. Add labels and formatting# 4. Display the plot# Your code here:

## Part C: Production ODE Solvers - scipy.integrate.solve_ivp

### Why Use scipy?
- **Adaptive step size**: Automatically adjusts $h$ for accuracy/efficiency
- **Multiple methods**: RK45, RK23, DOP853, BDF, LSODA, Radau
- **Error control**: Specify relative and absolute tolerances
- **Event detection**: Find when conditions are met (e.g., impact)
- **Dense output**: Smooth interpolation between steps
- **Robust**: Handles stiff systems, discontinuities

### Basic Usage
```python
sol = solve_ivp(fun, t_span, y0, method='RK45', 
                rtol=1e-6, atol=1e-9, dense_output=True)
```

### Method Selection Guide

| Method | Best For | Order | Notes |
|--------|----------|-------|-------|
| **RK45** | General purpose | 5 | Default, adaptive, excellent |
| **RK23** | Low accuracy needed | 3 | Faster for rough solutions |
| **DOP853** | High accuracy | 8 | Expensive but very accurate |
| **Radau** | Stiff systems | 5 | Implicit, handles stiffness |
| **BDF** | Very stiff | Varies | Backward differentiation |
| **LSODA** | Unknown stiffness | Varies | Auto-switches |

### Example: Nonlinear Pendulum
Compare methods for a nonlinear pendulum (no small-angle approximation):
$$\ddot{\theta} + \frac{g}{L}\sin\theta = 0$$

In [ ]:
# Example 3: Nonlinear Pendulum with scipy solve_ivp
print("="*70)
print("Example 3: scipy solve_ivp for Nonlinear Pendulum")
print("="*70)

# Pendulum parameters
L = 1.0        # m
g = 9.81       # m/s²
theta0 = np.pi/3  # 60 degrees initial angle
omega0 = 0.0   # released from rest

print(f"\nNonlinear pendulum:")
print(f"Length: L = {L} m")
print(f"Initial angle: θ₀ = {np.degrees(theta0):.1f}°")
print(f"Initial velocity: ω₀ = {omega0} rad/s")

# ODE function
def pendulum_ode(t, y):
    """Nonlinear pendulum: [θ, ω] -> [dθ/dt, dω/dt]"""
    theta, omega = y
    dtheta_dt = omega
    domega_dt = -(g/L) * np.sin(theta)
    return [dtheta_dt, domega_dt]

# Initial state
y0 = [theta0, omega0]
t_span = (0, 10)
t_eval = np.linspace(0, 10, 500)

# Compare different methods
methods = ['RK23', 'RK45', 'DOP853']
colors = ['red', 'blue', 'green']
solutions = {}

print(f"\nSolving with different methods:")
for method in methods:
    start_time = time.time()
    sol = solve_ivp(pendulum_ode, t_span, y0, method=method, 
                    t_eval=t_eval, rtol=1e-8, atol=1e-11)
    elapsed = time.time() - start_time
    
    solutions[method] = sol
    print(f"{method:8s}: {sol.nfev:5d} function evals, {elapsed*1000:.2f} ms")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Angular position
ax = axes[0, 0]
for method, color in zip(methods, colors):
    sol = solutions[method]
    ax.plot(sol.t, np.degrees(sol.y[0]), color=color, linewidth=2, 
            label=f'{method} ({sol.nfev} evals)', alpha=0.7)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Angle (degrees)')
ax.set_title('Pendulum Angle vs Time')
ax.legend()
ax.grid(True, alpha=0.3)

# Phase portrait
ax = axes[0, 1]
for method, color in zip(methods, colors):
    sol = solutions[method]
    ax.plot(np.degrees(sol.y[0]), sol.y[1], color=color, linewidth=2, 
            label=method, alpha=0.7)
ax.set_xlabel('Angle (degrees)')
ax.set_ylabel('Angular Velocity (rad/s)')
ax.set_title('Phase Portrait')
ax.legend()
ax.grid(True, alpha=0.3)

# Energy conservation check
ax = axes[1, 0]
for method, color in zip(methods, colors):
    sol = solutions[method]
    # Total energy = kinetic + potential
    KE = 0.5 * (L**2) * sol.y[1]**2  # (1/2) I ω²
    PE = (L * g) * (1 - np.cos(sol.y[0]))  # mgL(1 - cos θ)
    E_total = KE + PE
    E0 = (L * g) * (1 - np.cos(theta0))
    energy_error = (E_total - E0) / E0 * 100
    
    ax.semilogy(sol.t, np.abs(energy_error), color=color, linewidth=2, 
                label=method, alpha=0.7)

ax.set_xlabel('Time (s)')
ax.set_ylabel('Energy Error (%)')
ax.set_title('Energy Conservation Error')
ax.legend()
ax.grid(True, alpha=0.3, which='both')

# Difference between methods (use DOP853 as reference)
ax = axes[1, 1]
ref_sol = solutions['DOP853']
ref_angle = ref_sol.y[0]

for method, color in zip(['RK23', 'RK45'], ['red', 'blue']):
    sol = solutions[method]
    angle_diff = np.abs(sol.y[0] - ref_angle)
    ax.semilogy(sol.t, np.degrees(angle_diff), color=color, linewidth=2, 
                label=f'{method} vs DOP853', alpha=0.7)

ax.set_xlabel('Time (s)')
ax.set_ylabel('Angle Difference (degrees)')
ax.set_title('Method Comparison (vs DOP853)')
ax.legend()
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('notebooks/Teacher/figs/u7_l1_scipy_solve_ivp.png', dpi=150, bbox_inches='tight')
plt.show()

# Demonstrate dense output
print(f"\n{'='*70}")
print("Dense Output Feature:")
print("="*70)

sol = solve_ivp(pendulum_ode, t_span, y0, method='RK45', 
                rtol=1e-8, atol=1e-11, dense_output=True)

# Query solution at arbitrary times
t_query = np.array([1.5, 3.7, 5.2, 8.9])
y_query = sol.sol(t_query)

print(f"\nSolution at arbitrary times (continuous interpolation):")
for i, t in enumerate(t_query):
    theta = np.degrees(y_query[0, i])
    omega = y_query[1, i]
    print(f"t = {t:.1f} s: θ = {theta:7.3f}°, ω = {omega:7.4f} rad/s")

print(f"\n{'='*70}")
print("Best Practices:")
print("• Use RK45 as default for most problems")
print("• Adjust rtol/atol for accuracy requirements")
print("• Enable dense_output for smooth interpolation")
print("• Check energy conservation for validation")
print("• Use DOP853 for high-accuracy applications")
print("="*70)
# TODO: Use these parameters in your solutions below


## Summary

### Method Comparison

| Method | Order | Stability | Use Case |
|--------|-------|-----------|----------|
| **Euler** | 1 | Poor | Educational only |
| **RK2** | 2 | Good | Quick prototypes |
| **RK4** | 4 | Excellent | Manual implementation |
| **RK45** | 5 | Excellent | Production (scipy) |
| **DOP853** | 8 | Excellent | High accuracy |

### Key Takeaways

1. **Order matters**: Higher-order methods dramatically reduce error
2. **Adaptive stepping**: scipy automatically adjusts step size for efficiency
3. **Method selection**: RK45 is excellent default, DOP853 for high accuracy
4. **Validation**: Always check energy conservation (when applicable)
5. **Efficiency**: RK4 is most efficient for typical accuracy (if manual implementation needed)

### Practical Guidelines

**For simple systems** (manual implementation):
- Use RK4 with $h \approx T/100$ where $T$ is the period/timescale

**For production code** (scipy):
```python
sol = solve_ivp(ode_func, t_span, y0, method='RK45',
                rtol=1e-6, atol=1e-9, dense_output=True)
```

**For stiff systems** (rapid transients):
- Use `method='Radau'` or `method='LSODA'`

**For high accuracy** (e.g., orbital mechanics):
- Use `method='DOP853'` with tight tolerances

### Next Steps
- **Lecture 2**: Multi-body system simulation with constraints
- **Practical**: Implement complete simulation pipeline
- **Applications**: Robot control, vehicle dynamics, mechanism design